In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("internet_fijo_elt.bronze.conexiones_internet_fijo")
display(df_bronze.limit(10))

In [0]:
display(df_bronze.groupBy("Tecnología").count().orderBy(F.desc("count")))

In [0]:
display(df_bronze.groupBy("Segmento").count().orderBy(F.desc("count")))

In [0]:
display(df_bronze.groupBy("Periodo").count().orderBy("Periodo"))

In [0]:
df_bronze.count()

In [0]:
catalog = "internet_fijo_elt"
schema = "silver"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [0]:
df_silver = (
    df_bronze
    .withColumnRenamed("Periodo", "periodo")
    .withColumnRenamed("Empresa", "empresa")
    .withColumnRenamed("Departamento", "departamento")
    .withColumnRenamed("Provincia", "provincia")
    .withColumnRenamed("Distrito", "distrito")
    .withColumnRenamed("Tecnología", "tecnologia")
    .withColumnRenamed("Segmento", "segmento")
    .withColumnRenamed("Conexiones", "conexiones")
)

In [0]:
text_columns = ["empresa", "departamento", "provincia", "distrito", "tecnologia", "segmento"]

for col_name in text_columns:
    df_silver = df_silver.withColumn(col_name, F.trim(F.col(col_name)))

In [0]:
df_silver = df_silver.withColumn("tecnologia", F.regexp_replace(F.col("tecnologia"), r"^\d+\)\s*", ""))

In [0]:
df_silver = df_silver.withColumn("segmento", F.initcap(F.lower(F.col("segmento"))))

In [0]:
df_silver = df_silver.withColumn("periodo", F.trunc(F.to_date(F.col("periodo")), "month"))

In [0]:
df_silver = df_silver.withColumn("conexiones", F.col("conexiones").cast("long"))

In [0]:
df_silver = df_silver.filter(
    F.col("periodo").isNotNull() &
    F.col("empresa").isNotNull() &
    F.col("departamento").isNotNull() &
    F.col("provincia").isNotNull() &
    F.col("distrito").isNotNull() &
    F.col("tecnologia").isNotNull() &
    F.col("segmento").isNotNull() &
    F.col("conexiones").isNotNull()
)

In [0]:
df_silver = df_silver.filter(F.col("conexiones") >= 0)

In [0]:
print("Registros Silver:", df_silver.count())

In [0]:
display(df_silver.groupBy("tecnologia").count().orderBy(F.desc("count")))

In [0]:
display(df_silver.groupBy("segmento").count().orderBy(F.desc("count")))

In [0]:
display(df_silver.groupBy("periodo").count().orderBy("periodo"))

In [0]:
silver_table = "internet_fijo_elt.silver.conexiones_internet_fijo"

df_silver.write.format("delta").mode("overwrite").saveAsTable(silver_table)

In [0]:
spark.table("internet_fijo_elt.silver.conexiones_internet_fijo").count()